In [1]:
import time
SESSION_STARTED = time.monotonic()
!pip install -q transformers sentencepiece x-transformers optuna==4.6.0 scikit-learn
!apt-get -qq update
!apt-get -qq install -y unrar


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.1/143.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.4 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# XLM-R + x-transformers — Thai Depression — Strong Optimized

เวอร์ชันนี้เปลี่ยนเฉพาะ encoder เป็น `xlm-roberta-base` และปรับ data loader ให้ตรงกับ **Thai Depression dataset ที่ผู้ใช้อัปโหลด**

- ใช้ split ที่ dataset ให้มาโดยตรง: `train.json`, `valid.json`, `test.json`
- ไม่ทำ split ใหม่
- Label: `depression`, `no_depression`
- คง `MAX_LEN=128`, `EPOCHS=100`, `BATCH_SIZE=64`
- คง Optuna Strong Optimized framework เดิม: `N_TRIALS=25`, `OPTUNA_EPOCHS=35`
- encoder frozen
- decoder, pooling, classifier, loss search, threshold tuning และ objective คงเดิม


## Kaggle: latest checkpoint and epoch resume

- Save Version -> Save & Run All. Run until complete or Kaggle's 12-hour limit.
- Every completed epoch overwrites /kaggle/working/checkpoint_latest.pt.
- The checkpoint includes the active trial, completed trials, trainable weights,
  optimizer, scheduler, AMP scaler, and RNG states. The frozen encoder is reloaded.
- Next session: Add Input -> select the previous saved Notebook Output (or upload
  your own checkpoint as a private Dataset), then Save & Run All again.
- A single matching checkpoint_latest.pt under /kaggle/input is loaded automatically.
  Set RESUME_FROM below if more than one matches. Keep the dataset and settings unchanged.
- The resumed run repeats any epoch that was interrupted before its checkpoint finished.
- Final 100-epoch training and reports run only after 25 trials are complete.
- A hard platform shutdown may prevent Output publication; periodic checkpoints
  cannot guarantee recovery unless the saved file remains available.


In [2]:
import os
import json
import random
import warnings
import subprocess
import shutil
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss,
    multilabel_confusion_matrix
)

import optuna


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

DEVICE: cuda


In [5]:
MODEL_NAME = "xlm-roberta-base"

MAX_LEN = 128
EPOCHS = 100
BATCH_SIZE = 64
NUM_LABELS = 2

N_TRIALS = 25
OPTUNA_EPOCHS = 35

LABEL_COLUMNS = [
    "depression",
    "no_depression"
]

TEXT_COLUMN = "text"

# First run: leave empty. Next run: auto-find your checkpoint in attached Input.
RESUME_FROM = ""
# Example: RESUME_FROM = "/kaggle/input/my-backup/checkpoint_latest.pt"
SESSION_BUDGET_HOURS = 12.0


In [6]:
from pathlib import Path
import shutil

files = list(Path("/kaggle/input").rglob("checkpoint_latest.pt"))
assert len(files) == 1, f"ต้องมี checkpoint ใน Input เพียงไฟล์เดียว พบ: {files}"

RESUME_FROM = str(files[0])
shutil.copy2(RESUME_FROM, "/kaggle/working/checkpoint_latest.pt")
print("ใช้ checkpoint จาก:", RESUME_FROM)

ใช้ checkpoint จาก: /kaggle/input/datasets/abdulhakimsamae/checkpoint-latest1/checkpoint_latest.pt


In [7]:
# Latest checkpoint helpers
import time
import os
import shutil

WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WORK_DIR / "checkpoint_latest.pt"

if not CHECKPOINT_PATH.exists():
    if RESUME_FROM:
        source_checkpoint = Path(RESUME_FROM)
    else:
        candidates = sorted(Path("/kaggle/input").rglob("checkpoint_latest.pt"))
        if len(candidates) > 1:
            raise ValueError("Multiple checkpoints found; set RESUME_FROM to the intended file")
        source_checkpoint = candidates[0] if candidates else None
    if source_checkpoint is not None:
        shutil.copy2(source_checkpoint, CHECKPOINT_PATH)

CHECKPOINT_CONFIG = dict(
    format=1, optuna_version=optuna.__version__, model_name=MODEL_NAME,
    max_len=MAX_LEN, batch_size=BATCH_SIZE, epochs=EPOCHS,
    optuna_epochs=OPTUNA_EPOCHS, n_trials=N_TRIALS, labels=LABEL_COLUMNS
)


class SessionTimeLimit(Exception):
    pass


def session_expired():
    return time.monotonic() - SESSION_STARTED >= SESSION_BUDGET_HOURS * 3600


def capture_rng():
    return dict(python=random.getstate(), numpy=np.random.get_state(),
                torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda"]])


# Load only your own checkpoint from this notebook: it contains Python objects.
if CHECKPOINT_PATH.exists():
    CHECKPOINT = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    if CHECKPOINT.get("config") != CHECKPOINT_CONFIG:
        raise ValueError("Checkpoint configuration/version differs; use the same configuration and Optuna version")
    study = CHECKPOINT["study"]
    restore_rng(CHECKPOINT["rng"])
    print("Loaded:", CHECKPOINT_PATH)
else:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True)
    )
    CHECKPOINT = dict(config=CHECKPOINT_CONFIG, study=study,
                      active_trial=None, training=None)

SESSION_STOPPED = False
FINAL_TRAINING_COMPLETE = False


def save_latest():
    CHECKPOINT["rng"] = capture_rng()
    temporary = CHECKPOINT_PATH.with_suffix(".tmp")
    try:
        torch.save(CHECKPOINT, temporary)
        os.replace(temporary, CHECKPOINT_PATH)
    finally:
        temporary.unlink(missing_ok=True)
    # Human-readable results are overwritten too; the .pt is the resume file.
    study.trials_dataframe().to_csv(WORK_DIR / "trials_latest.csv", index=False)


def save_training(model, optimizer, scheduler, scaler, tag, params,
                  completed_epochs, epochs, steps_per_epoch):
    CHECKPOINT["training"] = dict(
        tag=tag, params=dict(params or {}), completed_epochs=completed_epochs,
        epochs=epochs, steps_per_epoch=steps_per_epoch,
        # Frozen encoder weights are reloaded from MODEL_NAME on the next run.
        model={k: v for k, v in model.state_dict().items() if not k.startswith("encoder.")},
        optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
        scaler=scaler.state_dict(), rng=capture_rng()
    )
    save_latest()
    print(f"Saved {tag}: epoch {completed_epochs}/{epochs}", flush=True)


Loaded: /kaggle/working/checkpoint_latest.pt


In [8]:
# ถ้าใช้ไฟล์ .rar ที่อัปโหลดมาโดยตรงใน Google Colab
# cell นี้จะ extract อัตโนมัติ
RAR_PATH = "/content/Thai Depression.rar"
EXTRACT_ROOT = "/content/thai_depression_extracted"

# รองรับกรณีอัปโหลดไฟล์ JSON ที่ extract มาแล้ว
TRAIN_JSON = "/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/train.json"
VALID_JSON = "/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/valid.json"
TEST_JSON = "/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/test.json"

if not (
    os.path.exists(TRAIN_JSON)
    and os.path.exists(VALID_JSON)
    and os.path.exists(TEST_JSON)
):
    if os.path.exists(RAR_PATH):
        os.makedirs(EXTRACT_ROOT, exist_ok=True)

        if shutil.which("unrar") is None:
            raise RuntimeError(
                "ไม่พบคำสั่ง unrar กรุณารัน cell ติดตั้งด้านบนก่อน "
                "หรือ extract Thai Depression.rar แล้วอัปโหลด train.json, valid.json, test.json"
            )

        subprocess.run(
            ["unrar", "x", "-o+", RAR_PATH, EXTRACT_ROOT + "/"],
            check=True
        )

        TRAIN_JSON = os.path.join(EXTRACT_ROOT, "Thai Depression", "train.json")
        VALID_JSON = os.path.join(EXTRACT_ROOT, "Thai Depression", "valid.json")
        TEST_JSON = os.path.join(EXTRACT_ROOT, "Thai Depression", "test.json")

# fallback สำหรับ local/sandbox
if not os.path.exists(TRAIN_JSON):
    TRAIN_JSON = "/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/train.json"
    VALID_JSON = "/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/valid.json"
    TEST_JSON = "/kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/test.json"

print("TRAIN JSON:", TRAIN_JSON)
print("VALID JSON:", VALID_JSON)
print("TEST JSON :", TEST_JSON)


def load_thai_depression_json(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    frame = pd.DataFrame(data, columns=[TEXT_COLUMN, "label"])

    frame[TEXT_COLUMN] = frame[TEXT_COLUMN].astype(str).str.strip()
    frame["label"] = frame["label"].astype(str).str.strip()

    unknown = sorted(set(frame["label"]) - set(LABEL_COLUMNS))
    if unknown:
        raise ValueError(f"พบ label ที่ไม่รู้จัก: {unknown}")

    for label in LABEL_COLUMNS:
        frame[label] = (frame["label"] == label).astype(int)

    return frame[[TEXT_COLUMN] + LABEL_COLUMNS]


train_df = load_thai_depression_json(TRAIN_JSON)
val_df = load_thai_depression_json(VALID_JSON)
test_df = load_thai_depression_json(TEST_JSON)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

print("\nTRAIN LABEL DISTRIBUTION")
print(train_df[LABEL_COLUMNS].sum())

print("\nVALIDATION LABEL DISTRIBUTION")
print(val_df[LABEL_COLUMNS].sum())

print("\nTEST LABEL DISTRIBUTION")
print(test_df[LABEL_COLUMNS].sum())

print("\nLABEL CARDINALITY")
print("TRAIN")
print(train_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())
print("VALIDATION")
print(val_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())
print("TEST")
print(test_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())


TRAIN JSON: /kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/train.json
VALID JSON: /kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/valid.json
TEST JSON : /kaggle/input/datasets/abdulhakimsamae/thai-depression/Thai Depression/test.json
TRAIN: 25077
VALIDATION: 3344
TEST: 5015

TRAIN LABEL DISTRIBUTION
depression       12837
no_depression    12240
dtype: int64

VALIDATION LABEL DISTRIBUTION
depression       1712
no_depression    1632
dtype: int64

TEST LABEL DISTRIBUTION
depression       2567
no_depression    2448
dtype: int64

LABEL CARDINALITY
TRAIN
1    25077
Name: count, dtype: int64
VALIDATION
1    3344
Name: count, dtype: int64
TEST
1    5015
Name: count, dtype: int64


In [9]:
# Thai Depression มี train/validation/test split มาให้แล้ว
# จึงไม่ split ใหม่ เพื่อรักษา protocol ของ dataset เดิม

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

assert (train_df[LABEL_COLUMNS].sum(axis=1) == 1).all()
assert (val_df[LABEL_COLUMNS].sum(axis=1) == 1).all()
assert (test_df[LABEL_COLUMNS].sum(axis=1) == 1).all()

print("ใช้ original dataset split โดยตรงเรียบร้อย")


ใช้ original dataset split โดยตรงเรียบร้อย


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        labels = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [13]:
# pos_weight เดิมทำให้ recall สูง แต่ precision ตกได้
# จึงไม่ fix pos_weight ตายตัว แต่ให้ Optuna tune pos_weight_scale
# scale = 0.0 คือไม่ใช้ pos_weight
# scale = 1.0 คือใช้ pos_weight เต็มสูตร negative / positive

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)
total_samples = len(train_df)
neg_counts = total_samples - label_counts

pos_weights_raw = neg_counts / (label_counts + 1e-6)

pos_weights_raw = torch.tensor(
    pos_weights_raw,
    dtype=torch.float
).to(device)

def build_pos_weight(pos_weight_scale):
    pos_weight_scale = float(pos_weight_scale)

    weights = 1.0 + (pos_weights_raw - 1.0) * pos_weight_scale

    return weights.to(device)

print("LABEL COUNTS:", label_counts)
print("RAW POS WEIGHTS:", pos_weights_raw)
print("NO POS WEIGHT:", build_pos_weight(0.0))
print("HALF POS WEIGHT:", build_pos_weight(0.5))


LABEL COUNTS: [12837. 12240.]
RAW POS WEIGHTS: tensor([0.9535, 1.0488], device='cuda:0')
NO POS WEIGHT: tensor([1., 1.], device='cuda:0')
HALF POS WEIGHT: tensor([0.9767, 1.0244], device='cuda:0')


In [14]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [15]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.15,
        ff_dropout=0.15,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # Fixed Thai encoder: train เฉพาะ decoder + pooling + classifier
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(hidden_size)

        self.dropout = nn.Dropout(0.30)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(self, input_ids, attention_mask):

        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits


In [16]:
def optimize_thresholds(
    y_true,
    y_probs,
    min_thr=0.20,
    max_thr=0.80,
    step=0.005
):

    thresholds = np.arange(
        min_thr,
        max_thr + 1e-9,
        step
    )

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_score = -1
        best_precision = -1

        true_support = y_true[:, i].sum()

        for thr in thresholds:

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            precision = precision_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            # ถ้า F1 เท่ากัน ให้เลือก threshold ที่ precision สูงกว่า
            # ช่วยแก้ปัญหา optimized เดิมที่ recall สูงแต่ precision ต่ำ
            if (score > best_score) or (
                np.isclose(score, best_score) and precision > best_precision
            ):

                best_score = score
                best_precision = precision
                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)


In [17]:
def evaluate(model, loader, thresholds=None):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    if thresholds is None:
        thresholds = np.array([0.5] * NUM_LABELS)

    preds = (all_probs >= thresholds).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    per_label_f1 = f1_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_label_precision = precision_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_label_recall = recall_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    subset_accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "per_label_f1": per_label_f1,
        "per_label_precision": per_label_precision,
        "per_label_recall": per_label_recall,
        "min_per_label_f1": float(np.min(per_label_f1)),
        "accuracy": subset_accuracy,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }


def objective_score(result):
    # objective ใหม่:
    # - macro_f1 สำคัญที่สุด เพราะต้องชนะราย class
    # - micro_f1 ยังสำคัญ
    # - min_per_label_f1 กันไม่ให้ class ใด class หนึ่งต่ำ โดยเฉพาะ Disgust
    # - subset accuracy ช่วยให้ทายชุด label ตรงขึ้น
    return (
        0.45 * result["macro_f1"]
        + 0.25 * result["micro_f1"]
        + 0.20 * result["min_per_label_f1"]
        + 0.10 * result["subset_accuracy"]
    )


In [18]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    epochs=None,
    pos_weight_scale=0.0,
    label_smoothing=0.0,
    loss_name="bce",
    focal_gamma=2.0,
    verbose=True,
    checkpoint_tag=None,
    checkpoint_params=None
):

    if epochs is None:
        epochs = EPOCHS

    pos_weight = build_pos_weight(pos_weight_scale)

    bce_loss = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight,
        reduction="none"
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * epochs

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps
    )

    use_amp = device.type == "cuda"

    scaler = torch.amp.GradScaler("cuda",
        enabled=use_amp
    )

    start_epoch = 0
    saved = CHECKPOINT.get("training")
    if checkpoint_tag is not None and saved and saved["tag"] == checkpoint_tag:
        if (saved["params"] != dict(checkpoint_params or {})
                or saved["epochs"] != epochs
                or saved["steps_per_epoch"] != len(train_loader)):
            raise ValueError("Training parameters or dataset batch count differ from checkpoint")
        incompatible = model.load_state_dict(saved["model"], strict=False)
        if incompatible.unexpected_keys or any(
            not key.startswith("encoder.") for key in incompatible.missing_keys
        ):
            raise ValueError("Checkpoint does not match the trainable model")
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
        restore_rng(saved["rng"])
        start_epoch = saved["completed_epochs"]
        print(f"Resuming {checkpoint_tag}: next epoch {start_epoch + 1}/{epochs}")
    elif checkpoint_tag is not None:
        save_training(model, optimizer, scheduler, scaler, checkpoint_tag,
                      checkpoint_params, 0, epochs, len(train_loader))

    for epoch in range(start_epoch, epochs):
        if session_expired():
            raise SessionTimeLimit()

        model.train()
        total_loss = 0

        loop = tqdm(
            train_loader,
            disable=not verbose
        )

        for batch in loop:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            if label_smoothing > 0:
                # smooth toward 0.5 for multi-label BCE
                labels_for_loss = labels * (1.0 - label_smoothing) + 0.5 * label_smoothing
            else:
                labels_for_loss = labels

            optimizer.zero_grad()

            with torch.amp.autocast("cuda", enabled=use_amp):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss_matrix = bce_loss(
                    logits,
                    labels_for_loss
                )

                if loss_name == "focal":
                    probs = torch.sigmoid(logits)
                    pt = probs * labels + (1 - probs) * (1 - labels)
                    focal_weight = (1 - pt).pow(focal_gamma)
                    loss = (focal_weight * loss_matrix).mean()
                else:
                    loss = loss_matrix.mean()

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

            if verbose:
                loop.set_description(
                    f"Epoch {epoch+1}/{epochs}"
                )

                loop.set_postfix(
                    loss=loss.item()
                )

        if verbose:

            val_raw = evaluate(
                model,
                val_loader
            )

            val_thresholds = optimize_thresholds(
                val_raw["labels"],
                val_raw["probs"]
            )

            val_result = evaluate(
                model,
                val_loader,
                val_thresholds
            )

            print("\n======================")
            print(f"Epoch {epoch+1}")
            print("======================")

            print(
                f"VAL SUBSET ACC : {val_result['subset_accuracy']:.4f}"
            )

            print(
                f"VAL MICRO P/R/F1: "
                f"{val_result['micro_precision']:.4f} / "
                f"{val_result['micro_recall']:.4f} / "
                f"{val_result['micro_f1']:.4f}"
            )

            print(
                f"VAL MACRO P/R/F1: "
                f"{val_result['macro_precision']:.4f} / "
                f"{val_result['macro_recall']:.4f} / "
                f"{val_result['macro_f1']:.4f}"
            )

            print(
                f"VAL MIN LABEL F1: {val_result['min_per_label_f1']:.4f}"
            )

            print(
                f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
            )

            print(
                f"VAL HAMMING LOSS: {val_result['hamming_loss']:.4f}"
            )

        if checkpoint_tag is not None:
            save_training(model, optimizer, scheduler, scaler, checkpoint_tag,
                          checkpoint_params, epoch + 1, epochs, len(train_loader))

    return model


In [19]:
def objective(trial):

    # baseline trial จะถูก enqueue ไว้ก่อนหน้า
    depth = trial.suggest_int(
        "depth",
        1,
        3
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4, 8]
    )

    # ลด dropout range จากเดิมที่สูงเกินไป เพราะเดิม optimized มีแนวโน้ม underfit / recall เยอะ precision ตก
    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.05,
        0.30
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.05,
        0.30
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-2,
        log=True
    )

    # จุดสำคัญ: ให้ Optuna เลือกว่าจะใช้ pos_weight แค่ไหน
    # ถ้า scale สูงเกินจะ recall สูงแต่ precision ลด
    pos_weight_scale = trial.suggest_float(
        "pos_weight_scale",
        0.0,
        0.60
    )

    label_smoothing = trial.suggest_float(
        "label_smoothing",
        0.0,
        0.04
    )

    loss_name = trial.suggest_categorical(
        "loss_name",
        ["bce", "focal"]
    )

    focal_gamma = trial.suggest_float(
        "focal_gamma",
        1.0,
        2.5
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr=lr,
        weight_decay=weight_decay,
        epochs=OPTUNA_EPOCHS,
        pos_weight_scale=pos_weight_scale,
        label_smoothing=label_smoothing,
        loss_name=loss_name,
        focal_gamma=focal_gamma,
        verbose=False,
        checkpoint_tag=f"trial:{trial.number}",
        checkpoint_params=dict(trial.params)
    )

    val_raw = evaluate(
        model,
        val_loader
    )

    thresholds = optimize_thresholds(
        val_raw["labels"],
        val_raw["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        thresholds
    )

    score = objective_score(val_result)

    trial.set_user_attr("val_macro_f1", float(val_result["macro_f1"]))
    trial.set_user_attr("val_micro_f1", float(val_result["micro_f1"]))
    trial.set_user_attr("val_min_per_label_f1", float(val_result["min_per_label_f1"]))
    trial.set_user_attr("val_subset_accuracy", float(val_result["subset_accuracy"]))
    trial.set_user_attr("thresholds", thresholds.tolist())

    return score


In [20]:
# Resume Optuna and the active trial
if not study.trials:
    study.enqueue_trial({
        "depth": 1, "heads": 2,
        "attn_dropout": 0.20, "ff_dropout": 0.20,
        "ff_mult": 2, "lr": 2e-5, "weight_decay": 1e-2,
        "pos_weight_scale": 0.0, "label_smoothing": 0.0,
        "loss_name": "bce", "focal_gamma": 2.0
    })
    save_latest()

try:
    while len(study.get_trials(states=[optuna.trial.TrialState.COMPLETE])) < N_TRIALS:
        if session_expired():
            raise SessionTimeLimit()
        trial = CHECKPOINT["active_trial"]
        if trial is None:
            trial = study.ask()
            CHECKPOINT["active_trial"] = trial
            CHECKPOINT["training"] = None
            save_latest()
        print("Trial:", trial.number, "parameters:", trial.params, flush=True)
        score = objective(trial)
        study.tell(trial, score)
        CHECKPOINT["active_trial"] = None
        CHECKPOINT["training"] = None
        save_latest()
        print(f"Trial {trial.number} complete: {score:.6f}", flush=True)
except SessionTimeLimit:
    SESSION_STOPPED = True
    print("Session budget reached. The latest epoch is saved; final training is skipped.")

SEARCH_COMPLETE = len(study.get_trials(states=[optuna.trial.TrialState.COMPLETE])) >= N_TRIALS
print("Completed trials:", len(study.get_trials(states=[optuna.trial.TrialState.COMPLETE])), "/", N_TRIALS)


Trial: 16 parameters: {'depth': 2, 'heads': 4, 'attn_dropout': 0.2155337515666265, 'ff_dropout': 0.28236296878717765, 'ff_mult': 4, 'lr': 5.495086510940928e-05, 'weight_decay': 4.025059194852571e-06, 'pos_weight_scale': 0.23546028957228188, 'label_smoothing': 0.02110992012369271, 'loss_name': 'focal', 'focal_gamma': 1.004543033682979}


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resuming trial:16: next epoch 31/35
Saved trial:16: epoch 31/35
Saved trial:16: epoch 32/35
Saved trial:16: epoch 33/35
Saved trial:16: epoch 34/35
Saved trial:16: epoch 35/35
Trial 16 complete: 0.792398
Trial: 17 parameters: {}


...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved trial:23: epoch 0/35
Saved trial:23: epoch 1/35
Saved trial:23: epoch 2/35
Saved trial:23: epoch 3/35
Saved trial:23: epoch 4/35
Saved trial:23: epoch 5/35
Saved trial:23: epoch 6/35
Saved trial:23: epoch 7/35
Saved trial:23: epoch 8/35
Saved trial:23: epoch 9/35
Saved trial:23: epoch 10/35
Saved trial:23: epoch 11/35
Saved trial:23: epoch 12/35
Saved trial:23: epoch 13/35
Saved trial:23: epoch 14/35
Saved trial:23: epoch 15/35
Saved trial:23: epoch 16/35
Saved trial:23: epoch 17/35
Saved trial:23: epoch 18/35
Saved trial:23: epoch 19/35
Saved trial:23: epoch 20/35
Saved trial:23: epoch 21/35
Saved trial:23: epoch 22/35
Saved trial:23: epoch 23/35
Saved trial:23: epoch 24/35
Saved trial:23: epoch 25/35
Saved trial:23: epoch 26/35
Saved trial:23: epoch 27/35
Saved trial:23: epoch 28/35
Saved trial:23: epoch 29/35
Saved trial:23: epoch 30/35
Saved trial:23: epoch 31/35
Saved trial:23: epoch 32/35
Saved trial:23: epoch 33/35
Saved trial:23: epoch 34/35
Saved trial:23: epoch 35/35
Tr

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved trial:24: epoch 0/35
Saved trial:24: epoch 1/35
Saved trial:24: epoch 2/35
Saved trial:24: epoch 3/35
Saved trial:24: epoch 4/35
Saved trial:24: epoch 5/35
Saved trial:24: epoch 6/35
Saved trial:24: epoch 7/35
Saved trial:24: epoch 8/35
Saved trial:24: epoch 9/35
Saved trial:24: epoch 10/35
Saved trial:24: epoch 11/35
Saved trial:24: epoch 12/35
Saved trial:24: epoch 13/35
Saved trial:24: epoch 14/35
Saved trial:24: epoch 15/35
Saved trial:24: epoch 16/35
Saved trial:24: epoch 17/35
Saved trial:24: epoch 18/35
Saved trial:24: epoch 19/35
Saved trial:24: epoch 20/35
Saved trial:24: epoch 21/35
Saved trial:24: epoch 22/35
Saved trial:24: epoch 23/35
Saved trial:24: epoch 24/35
Saved trial:24: epoch 25/35
Saved trial:24: epoch 26/35
Saved trial:24: epoch 27/35
Saved trial:24: epoch 28/35
Saved trial:24: epoch 29/35
Saved trial:24: epoch 30/35
Saved trial:24: epoch 31/35
Saved trial:24: epoch 32/35
Saved trial:24: epoch 33/35
Saved trial:24: epoch 34/35
Saved trial:24: epoch 35/35
Tr

In [21]:
if SEARCH_COMPLETE and not SESSION_STOPPED:
    print("\n======================")
    print("BEST PARAMETERS")
    print("======================")

    print(study.best_params)

    print("\n======================")
    print("BEST TRIAL VALIDATION ATTRS")
    print("======================")

    print("objective_score:", study.best_value)
    print("val_macro_f1:", study.best_trial.user_attrs.get("val_macro_f1"))
    print("val_micro_f1:", study.best_trial.user_attrs.get("val_micro_f1"))
    print("val_min_per_label_f1:", study.best_trial.user_attrs.get("val_min_per_label_f1"))
    print("val_subset_accuracy:", study.best_trial.user_attrs.get("val_subset_accuracy"))

    best_params = study.best_params



BEST PARAMETERS
{'depth': 2, 'heads': 4, 'attn_dropout': 0.13531448905077936, 'ff_dropout': 0.28901049420092895, 'ff_mult': 3, 'lr': 9.088180920175775e-05, 'weight_decay': 0.00018044458901954555, 'pos_weight_scale': 0.29207675834478736, 'label_smoothing': 0.023812574600130035, 'loss_name': 'bce', 'focal_gamma': 1.4936562099725805}

BEST TRIAL VALIDATION ATTRS
objective_score: 0.8007330100569677
val_macro_f1: 0.8101880133990084
val_micro_f1: 0.8102893890675241
val_min_per_label_f1: 0.8040943986352005
val_subset_accuracy: 0.7275717703349283


In [22]:
if SEARCH_COMPLETE and not SESSION_STOPPED:
    final_model = EmotionModel(
        depth=best_params["depth"],
        heads=best_params["heads"],
        attn_dropout=best_params["attn_dropout"],
        ff_dropout=best_params["ff_dropout"],
        ff_mult=best_params["ff_mult"]
    ).to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
if SEARCH_COMPLETE and not SESSION_STOPPED:
    try:
        final_model = train_model(
            final_model,
            train_loader,
            val_loader,
            lr=best_params["lr"],
            weight_decay=best_params["weight_decay"],
            epochs=EPOCHS,
            pos_weight_scale=best_params["pos_weight_scale"],
            label_smoothing=best_params["label_smoothing"],
            loss_name=best_params["loss_name"],
            focal_gamma=best_params["focal_gamma"],
            verbose=True,
            checkpoint_tag="final",
            checkpoint_params=dict(best_params)
        )
        FINAL_TRAINING_COMPLETE = True
    except SessionTimeLimit:
        SESSION_STOPPED = True
        print("Final training paused; latest epoch saved.")


Saved final: epoch 0/100


Epoch 1/100: 100%|██████████| 392/392 [01:06<00:00,  5.91it/s, loss=0.63]



Epoch 1
VAL SUBSET ACC : 0.1717
VAL MICRO P/R/F1: 0.5359 / 0.9566 / 0.6870
VAL MACRO P/R/F1: 0.5370 / 0.9568 / 0.6873
VAL MIN LABEL F1: 0.6651
VAL WEIGHTED F1: 0.6879
VAL HAMMING LOSS: 0.4359
Saved final: epoch 1/100


Epoch 2/100: 100%|██████████| 392/392 [01:06<00:00,  5.91it/s, loss=0.635]



Epoch 2
VAL SUBSET ACC : 0.6086
VAL MICRO P/R/F1: 0.6813 / 0.8367 / 0.7510
VAL MACRO P/R/F1: 0.6811 / 0.8366 / 0.7509
VAL MIN LABEL F1: 0.7390
VAL WEIGHTED F1: 0.7511
VAL HAMMING LOSS: 0.2774
Saved final: epoch 2/100


...


Epoch 98
VAL SUBSET ACC : 0.7524
VAL MICRO P/R/F1: 0.7744 / 0.8502 / 0.8105
VAL MACRO P/R/F1: 0.7744 / 0.8498 / 0.8103
VAL MIN LABEL F1: 0.8026
VAL WEIGHTED F1: 0.8105
VAL HAMMING LOSS: 0.1987
Saved final: epoch 98/100


Epoch 99/100: 100%|██████████| 392/392 [01:06<00:00,  5.92it/s, loss=0.432]



Epoch 99
VAL SUBSET ACC : 0.7001
VAL MICRO P/R/F1: 0.7472 / 0.8864 / 0.8108
VAL MACRO P/R/F1: 0.7475 / 0.8858 / 0.8105
VAL MIN LABEL F1: 0.8031
VAL WEIGHTED F1: 0.8107
VAL HAMMING LOSS: 0.2068
Saved final: epoch 99/100


Epoch 100/100: 100%|██████████| 392/392 [01:06<00:00,  5.89it/s, loss=0.355]



Epoch 100
VAL SUBSET ACC : 0.7270
VAL MICRO P/R/F1: 0.7606 / 0.8675 / 0.8106
VAL MACRO P/R/F1: 0.7627 / 0.8665 / 0.8101
VAL MIN LABEL F1: 0.8024
VAL WEIGHTED F1: 0.8102
VAL HAMMING LOSS: 0.2028
Saved final: epoch 100/100


In [24]:
if FINAL_TRAINING_COMPLETE and not SESSION_STOPPED:
    val_result = evaluate(
        final_model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_result["labels"],
        val_result["probs"]
    )

    val_result_tuned = evaluate(
        final_model,
        val_loader,
        best_thresholds
    )

    print("\n======================")
    print("BEST THRESHOLDS FROM FINAL VALIDATION")
    print("======================")

    for label, thr in zip(
        LABEL_COLUMNS,
        best_thresholds
    ):
        print(f"{label}: {thr:.3f}")

    print("\nVALIDATION RESULT AFTER THRESHOLD TUNING")
    print("macro_f1:", val_result_tuned["macro_f1"])
    print("micro_f1:", val_result_tuned["micro_f1"])
    print("min_per_label_f1:", val_result_tuned["min_per_label_f1"])
    print("subset_accuracy:", val_result_tuned["subset_accuracy"])



BEST THRESHOLDS FROM FINAL VALIDATION
depression: 0.275
no_depression: 0.480

VALIDATION RESULT AFTER THRESHOLD TUNING
macro_f1: 0.8100602258734868
micro_f1: 0.8105616093880972
min_per_label_f1: 0.8023916292974589
subset_accuracy: 0.7269736842105263


In [25]:
if FINAL_TRAINING_COMPLETE and not SESSION_STOPPED:
    def summarize_result(name, result):

        return {
            "split": name,
            "subset_accuracy_exact_match": result["subset_accuracy"],
            "micro_precision": result["micro_precision"],
            "micro_recall": result["micro_recall"],
            "micro_f1": result["micro_f1"],
            "macro_precision": result["macro_precision"],
            "macro_recall": result["macro_recall"],
            "macro_f1": result["macro_f1"],
            "weighted_f1": result["weighted_f1"],
            "min_per_label_f1": result["min_per_label_f1"],
            "hamming_loss": result["hamming_loss"]
        }


    train_result = evaluate(
        final_model,
        train_eval_loader,
        best_thresholds
    )

    val_result_final = evaluate(
        final_model,
        val_loader,
        best_thresholds
    )

    test_result = evaluate(
        final_model,
        test_loader,
        best_thresholds
    )

    metrics_summary = pd.DataFrame([
        summarize_result("train", train_result),
        summarize_result("validation", val_result_final),
        summarize_result("test", test_result)
    ])

    print("\n======================")
    print("FINAL METRICS SUMMARY")
    print("======================")
    display(metrics_summary)

    print("\n======================")
    print("BEST THRESHOLDS")
    print("======================")

    threshold_df = pd.DataFrame({
        "label": LABEL_COLUMNS,
        "threshold": best_thresholds
    })

    display(threshold_df)

    print("\n======================")
    print("TEST SET MAIN RESULT")
    print("======================")

    print(f"Subset Accuracy / Exact Match : {test_result['subset_accuracy']:.4f}")
    print(f"Micro Precision              : {test_result['micro_precision']:.4f}")
    print(f"Micro Recall                 : {test_result['micro_recall']:.4f}")
    print(f"Micro F1                     : {test_result['micro_f1']:.4f}")
    print(f"Macro Precision              : {test_result['macro_precision']:.4f}")
    print(f"Macro Recall                 : {test_result['macro_recall']:.4f}")
    print(f"Macro F1                     : {test_result['macro_f1']:.4f}")
    print(f"Weighted F1                  : {test_result['weighted_f1']:.4f}")
    print(f"Min Per-label F1             : {test_result['min_per_label_f1']:.4f}")
    print(f"Hamming Loss                 : {test_result['hamming_loss']:.4f}")



FINAL METRICS SUMMARY


,split,subset_accuracy_exact_match,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,min_per_label_f1,hamming_loss
0,train,0.802488,0.823993,0.924672,0.871434,0.827898,0.923631,0.871416,0.871422,0.871148,0.136420
1,validation,0.726974,0.760619,0.867524,0.810562,0.762724,0.866467,0.810060,0.810244,0.802392,0.202751
2,test,0.719242,0.751982,0.851246,0.798541,0.755544,0.849818,0.797628,0.797898,0.786252,0.214756



BEST THRESHOLDS


,label,threshold
0,depression,0.275
1,no_depression,0.480



TEST SET MAIN RESULT
Subset Accuracy / Exact Match : 0.7192
Micro Precision              : 0.7520
Micro Recall                 : 0.8512
Micro F1                     : 0.7985
Macro Precision              : 0.7555
Macro Recall                 : 0.8498
Macro F1                     : 0.7976
Weighted F1                  : 0.7979
Min Per-label F1             : 0.7863
Hamming Loss                 : 0.2148


In [26]:
if FINAL_TRAINING_COMPLETE and not SESSION_STOPPED:
    print("\n======================")
    print("TEST CLASSIFICATION REPORT")
    print("======================")

    report_dict = classification_report(
        test_result["labels"],
        test_result["preds"],
        target_names=LABEL_COLUMNS,
        zero_division=0,
        output_dict=True
    )

    per_label_report = pd.DataFrame(report_dict).T
    display(per_label_report)

    per_label_metrics = pd.DataFrame({
        "label": LABEL_COLUMNS,
        "precision": test_result["per_label_precision"],
        "recall": test_result["per_label_recall"],
        "f1_score": test_result["per_label_f1"],
        "true_support": test_result["labels"].sum(axis=0).astype(int),
        "predicted_positive": test_result["preds"].sum(axis=0).astype(int),
        "threshold": best_thresholds
    })

    print("\n======================")
    print("PER-LABEL TEST METRICS")
    print("======================")
    display(per_label_metrics)

    print("\n======================")
    print("TEST MULTI-LABEL CONFUSION MATRIX")
    print("======================")

    mcm = multilabel_confusion_matrix(
        test_result["labels"],
        test_result["preds"]
    )

    confusion_rows = []

    for i, label in enumerate(LABEL_COLUMNS):
        tn, fp, fn, tp = mcm[i].ravel()

        confusion_rows.append({
            "label": label,
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn)
        })

    confusion_df = pd.DataFrame(confusion_rows)
    display(confusion_df)

    test_probs_df = pd.DataFrame(
        test_result["probs"],
        columns=[f"prob_{label}" for label in LABEL_COLUMNS]
    )

    test_preds_df = pd.DataFrame(
        test_result["preds"],
        columns=[f"pred_{label}" for label in LABEL_COLUMNS]
    )

    test_true_df = test_df[[TEXT_COLUMN] + LABEL_COLUMNS].copy()

    test_predictions_output = pd.concat(
        [
            test_true_df.reset_index(drop=True),
            test_probs_df.reset_index(drop=True),
            test_preds_df.reset_index(drop=True)
        ],
        axis=1
    )

    metrics_summary.to_csv(
        "/kaggle/working/xlmr_thai_depression_metrics_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    threshold_df.to_csv(
        "/kaggle/working/xlmr_thai_depression_thresholds.csv",
        index=False,
        encoding="utf-8-sig"
    )

    per_label_report.to_csv(
        "/kaggle/working/xlmr_thai_depression_per_label_report.csv",
        encoding="utf-8-sig"
    )

    per_label_metrics.to_csv(
        "/kaggle/working/xlmr_thai_depression_per_label_metrics.csv",
        index=False,
        encoding="utf-8-sig"
    )

    confusion_df.to_csv(
        "/kaggle/working/xlmr_thai_depression_confusion_matrix.csv",
        index=False,
        encoding="utf-8-sig"
    )

    test_predictions_output.to_csv(
        "/kaggle/working/xlmr_thai_depression_test_predictions.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\nEXPORTED FILES")
    print("/kaggle/working/xlmr_thai_depression_metrics_summary.csv")
    print("/kaggle/working/xlmr_thai_depression_thresholds.csv")
    print("/kaggle/working/xlmr_thai_depression_per_label_report.csv")
    print("/kaggle/working/xlmr_thai_depression_per_label_metrics.csv")
    print("/kaggle/working/xlmr_thai_depression_confusion_matrix.csv")
    print("/kaggle/working/xlmr_thai_depression_test_predictions.csv")



TEST CLASSIFICATION REPORT


,precision,recall,f1-score,support
depression,0.728180,0.910012,0.809004,2567.0
no_depression,0.782908,0.789624,0.786252,2448.0
micro avg,0.751982,0.851246,0.798541,5015.0
macro avg,0.755544,0.849818,0.797628,5015.0
weighted avg,0.754894,0.851246,0.797898,5015.0
samples avg,0.785244,0.851246,0.807245,5015.0



PER-LABEL TEST METRICS


,label,precision,recall,f1_score,true_support,predicted_positive,threshold
0,depression,0.728180,0.910012,0.809004,2567,3208,0.275
1,no_depression,0.782908,0.789624,0.786252,2448,2469,0.480



TEST MULTI-LABEL CONFUSION MATRIX


,label,TP,FP,FN,TN
0,depression,2336,872,231,1576
1,no_depression,1933,536,515,2031



EXPORTED FILES
/kaggle/working/xlmr_thai_depression_metrics_summary.csv
/kaggle/working/xlmr_thai_depression_thresholds.csv
/kaggle/working/xlmr_thai_depression_per_label_report.csv
/kaggle/working/xlmr_thai_depression_per_label_metrics.csv
/kaggle/working/xlmr_thai_depression_confusion_matrix.csv
/kaggle/working/xlmr_thai_depression_test_predictions.csv


In [27]:
if FINAL_TRAINING_COMPLETE and not SESSION_STOPPED:
    SAVE_PATH = "/kaggle/working/final_xlmr_thai_depression_xtransformers_strong_optimized.pt"

    torch.save(
        {
            "model_state_dict": final_model.state_dict(),
            "thresholds": best_thresholds,
            "best_params": best_params,
            "labels": LABEL_COLUMNS,
            "model_name": MODEL_NAME,
            "max_len": MAX_LEN,
            "dataset": "Thai Depression (original train/valid/test JSON)",
            "seed": SEED,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "optuna_trials": N_TRIALS,
            "optuna_epochs": OPTUNA_EPOCHS,
            "objective": "0.45 macro_f1 + 0.25 micro_f1 + 0.20 min_per_label_f1 + 0.10 subset_accuracy",
            "note": "Encoder comparison version; encoder frozen; same Strong Optimized EAQD/x-transformers framework."
        },
        SAVE_PATH
    )

    print("\nMODEL SAVED:", SAVE_PATH)



MODEL SAVED: /kaggle/working/final_xlmr_thai_depression_xtransformers_strong_optimized.pt


In [28]:

print("Resume file:", CHECKPOINT_PATH)
print("Latest trial results:", WORK_DIR / "trials_latest.csv")
saved = CHECKPOINT.get("training")
if saved:
    print("Saved stage:", saved["tag"], "completed epochs:", saved["completed_epochs"])
print("Attach this run's saved Output as Input in the next session to resume.")


Resume file: /kaggle/working/checkpoint_latest.pt
Latest trial results: /kaggle/working/trials_latest.csv
Saved stage: final completed epochs: 100
Attach this run's saved Output as Input in the next session to resume.
